# COFFEEBEAN — ANC Training on Google Colab
GPU-accelerated training with MLflow tracking to DagsHub

## How it works
1. Installs PyTorch (GPU) + MLflow + DVC
2. Clones repo from GitHub
3. Pulls dataset from Google Drive
4. Trains ANC model with real training loop
5. Logs all metrics to DagsHub MLflow (cloud)
6. Exports to ONNX and saves back to Drive

In [ ]:
# ── Parameters (injected by Airflow via papermill) ─────────────────────────
DAGSHUB_USERNAME = "Zenith1415"
DAGSHUB_TOKEN    = "144536a3b5a9c6e893d7bd140072c92a1cdd7a24"
REPO_NAME        = "COFFEEBEAN"
GITHUB_REPO      = "Zenith1415/COFFEEBEAN"
EPOCHS           = 50
BATCH_SIZE       = 32
LEARNING_RATE    = 0.001
COLAB_DRIVE_PATH = "/content/drive/MyDrive/COFFEEBEAN"

In [ ]:
# ── 1. Install dependencies ────────────────────────────────────────────────
!pip install torch torchaudio --index-url https://download.pytorch.org/whl/cu118 -q
!pip install mlflow dvc dvc-gdrive torchmetrics pystoi pesq onnx onnxruntime onnxscript pyyaml -q

In [ ]:
# ── 2. Mount Google Drive (for dataset) ────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')
print('Drive mounted')

In [ ]:
# ── 3. Clone repo and configure MLflow to DagsHub ─────────────────────────
import os

os.environ['MLFLOW_TRACKING_URI']      = f'https://dagshub.com/{DAGSHUB_USERNAME}/{REPO_NAME}.mlflow'
os.environ['MLFLOW_TRACKING_USERNAME'] = DAGSHUB_USERNAME
os.environ['MLFLOW_TRACKING_PASSWORD'] = DAGSHUB_TOKEN
os.environ['PYTHONUTF8']               = '1'

!git clone https://github.com/{GITHUB_REPO}.git /content/COFFEEBEAN 2>/dev/null || echo 'Already cloned'
%cd /content/COFFEEBEAN

# Verify DagsHub connection
import mlflow
mlflow.set_tracking_uri(os.environ['MLFLOW_TRACKING_URI'])
print(f'MLflow tracking: {os.environ["MLFLOW_TRACKING_URI"]}')
print('DagsHub connection OK')

In [ ]:
# ── 4. Pull dataset from Google Drive ─────────────────────────────────────
import shutil
from pathlib import Path

drive_data = Path(COLAB_DRIVE_PATH) / 'data'
if drive_data.exists():
    shutil.copytree(str(drive_data), 'data', dirs_exist_ok=True)
    wav_count = len(list(Path('data/raw').rglob('*.wav')))
    print(f'Dataset copied from Drive: {wav_count} WAV files')
else:
    print(f'Drive path not found: {drive_data}')
    print('Attempting DVC pull...')
    !dvc pull || echo 'DVC pull failed — add data manually'

In [ ]:
# ── 5. Check GPU ───────────────────────────────────────────────────────────
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')

In [ ]:
# ── 6. Override config for Colab session ──────────────────────────────────
import yaml
cfg = yaml.safe_load(open('configs/config.yaml'))
cfg['training']['epochs']        = EPOCHS
cfg['training']['batch_size']    = BATCH_SIZE
cfg['training']['learning_rate'] = LEARNING_RATE
with open('configs/config.yaml', 'w') as f:
    yaml.dump(cfg, f)
print('Config:', cfg['training'])

In [ ]:
# ── 7. Train ───────────────────────────────────────────────────────────────
import sys
sys.path.insert(0, '/content/COFFEEBEAN')
from src.training.train import train

run_id = train()
print(f'\nTraining complete. Run ID: {run_id}')
print(f'View at: https://dagshub.com/{DAGSHUB_USERNAME}/{REPO_NAME}.mlflow')

In [ ]:
# ── 8. Evaluate ────────────────────────────────────────────────────────────
from src.evaluation.run_evaluation import run_evaluation

results = run_evaluation(
    checkpoint='models/anc_checkpoint.pt',
    run_id=run_id,
    snr_levels=[-5.0, 0.0, 5.0, 10.0],
    num_pairs=10,
)
print('\nEvaluation results:')
for tag, metrics in results.items():
    print(f"  {tag}: SNR+={metrics.get('snr_improvement')}dB STOI={metrics.get('stoi_enhanced')} PESQ={metrics.get('pesq_enhanced')}")

In [ ]:
# ── 9. Export ONNX ─────────────────────────────────────────────────────────
from src.deployment.export import export_to_onnx, benchmark_onnx
from src.training.model import ANCAudioModel

model = ANCAudioModel(cfg)
model.load_state_dict(torch.load('models/anc_checkpoint.pt', map_location='cpu'))

export_info = export_to_onnx(model, 'models/anc_model.onnx', sample_rate=16000)
bench       = benchmark_onnx('models/anc_model.onnx', sample_rate=16000)
print('\nONNX Export:', export_info)
print('Benchmark:',   bench)

In [ ]:
# ── 10. Save artifacts to Drive ───────────────────────────────────────────
dest = Path(COLAB_DRIVE_PATH) / 'models'
dest.mkdir(parents=True, exist_ok=True)

for f in ['models/anc_checkpoint.pt', 'models/anc_model.onnx']:
    if Path(f).exists():
        shutil.copy(f, str(dest / Path(f).name))
        print(f'Saved {f} -> {dest}')

print(f'\nAll done! View experiments at:')
print(f'https://dagshub.com/{DAGSHUB_USERNAME}/{REPO_NAME}.mlflow')